In this notebook, we aim to:
- generate candidate examples for all five tasks,
- inspect the generated examples,
- run simple quality checks,
- manually approve or reject examples,
- export the reviewed candidate pool,
- select approved base examples for the next phase.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

In [2]:
# Make sure the project root is on the Python path for imports. This allows us to import from src/ and other directories regardless of whether we're running from the root or the notebooks/ directory.
if str(Path.cwd().parent) not in sys.path and Path.cwd().name == "notebooks":
    sys.path.append(str(Path.cwd().parent))
elif str(Path.cwd()) not in sys.path and Path.cwd().name != "notebooks":
    sys.path.append(str(Path.cwd()))

In [3]:
from src.generation import (
    generate_single_label_candidates,
    generate_multi_label_candidates,
    generate_ie_candidates,
    generate_transformation_candidates,
    generate_qa_candidates,
    generate_all_candidates,
    render_input_for_review,
    candidate_to_review_row,
    auto_flag_candidate,
    find_exact_duplicate_inputs,
    set_review_status,
    get_examples_by_status,
    select_final_base_examples,
    example_to_dict,
    save_candidates_to_jsonl,
)

In [4]:
# Generate candidate examples for each task
single_label_candidates = generate_single_label_candidates()
multi_label_candidates = generate_multi_label_candidates()
ie_candidates = generate_ie_candidates()
transformation_candidates = generate_transformation_candidates()
qa_candidates = generate_qa_candidates()

print("Single-label candidates:", len(single_label_candidates))
print("Multi-label candidates:", len(multi_label_candidates))
print("Information extraction candidates:", len(ie_candidates))
print("Transformation candidates:", len(transformation_candidates))
print("Extractive QA candidates:", len(qa_candidates))

Single-label candidates: 50
Multi-label candidates: 50
Information extraction candidates: 50
Transformation candidates: 50
Extractive QA candidates: 50


In [5]:
# Combine all candidate pools
all_candidates = generate_all_candidates()
print("Total candidates:", len(all_candidates))

Total candidates: 250


In [6]:
# View one example from each task

task_examples = {}

for ex in all_candidates:
    if ex.task_name not in task_examples:
        task_examples[ex.task_name] = ex

for task_name, ex in task_examples.items():
    print("=" * 100)
    print("TASK:", task_name)
    print("EXAMPLE ID:", ex.example_id)
    print("TEMPLATE:", ex.template_name)
    print("INSTRUCTION:", ex.instruction)
    print("INPUT:")
    print(render_input_for_review(ex))
    print("GOLD OUTPUT:", ex.gold_output)
    print()

TASK: single_label_classification
EXAMPLE ID: slc_000
TEMPLATE: product_review
INSTRUCTION: Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.
INPUT:
The medical software was excellent and worked smooth and satisfying.
GOLD OUTPUT: {'label': 'positive'}

TASK: multi_label_classification
EXAMPLE ID: mlc_050
TEMPLATE: institutional_announcement
INSTRUCTION: Assign all applicable topic labels from {finance, health, politics, sports, tech}. Return them alphabetically.
INPUT:
The government announced a new election reform related to election reform.
GOLD OUTPUT: {'labels': ['politics']}

TASK: information_extraction
EXAMPLE ID: ie_100
TEMPLATE: meeting_announcement
INSTRUCTION: Extract the fields person, date, and location from the text.
INPUT:
Alice Smith will speak in Rome on 2024-01-15.
GOLD OUTPUT: {'person': 'Alice Smith', 'date': '2024-01-15', 'location': 'Rome'}

TASK: rule_based_transformation
EXAMPLE ID: rbt_150
TEMPLATE: mixed_case_state

In [7]:
# Convert candidates to a review table 

review_rows = [candidate_to_review_row(ex) for ex in all_candidates]
review_df = pd.DataFrame(review_rows)

print("Review table shape:", review_df.shape)
review_df.head(10)

Review table shape: (250, 8)


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
0,slc_000,single_label_classification,product_review,Classify the sentiment of the text using exact...,The medical software was excellent and worked ...,"{""label"": ""positive""}",pending,
1,slc_001,single_label_classification,product_review,Classify the sentiment of the text using exact...,The robot vacuum was excellent and worked smoo...,"{""label"": ""positive""}",pending,
2,slc_002,single_label_classification,product_review,Classify the sentiment of the text using exact...,The budget phone was excellent and worked smoo...,"{""label"": ""positive""}",pending,
3,slc_003,single_label_classification,product_review,Classify the sentiment of the text using exact...,The fitness watch was excellent and worked smo...,"{""label"": ""positive""}",pending,
4,slc_004,single_label_classification,product_review,Classify the sentiment of the text using exact...,The solar battery was excellent and worked smo...,"{""label"": ""positive""}",pending,
5,slc_005,single_label_classification,product_review,Classify the sentiment of the text using exact...,The medical software was great and worked plea...,"{""label"": ""positive""}",pending,
6,slc_006,single_label_classification,product_review,Classify the sentiment of the text using exact...,The robot vacuum was great and worked pleasant...,"{""label"": ""positive""}",pending,
7,slc_007,single_label_classification,product_review,Classify the sentiment of the text using exact...,The budget phone was great and worked pleasant...,"{""label"": ""positive""}",pending,
8,slc_008,single_label_classification,product_review,Classify the sentiment of the text using exact...,The fitness watch was great and worked pleasan...,"{""label"": ""positive""}",pending,
9,slc_009,single_label_classification,product_review,Classify the sentiment of the text using exact...,The solar battery was great and worked pleasan...,"{""label"": ""positive""}",pending,


In [8]:
# Inspect a few examples from each task

for task_name in review_df["task_name"].unique():
    print("- " + task_name)
    display(review_df[review_df["task_name"] == task_name].head(5))

- single_label_classification


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
0,slc_000,single_label_classification,product_review,Classify the sentiment of the text using exact...,The medical software was excellent and worked ...,"{""label"": ""positive""}",pending,
1,slc_001,single_label_classification,product_review,Classify the sentiment of the text using exact...,The robot vacuum was excellent and worked smoo...,"{""label"": ""positive""}",pending,
2,slc_002,single_label_classification,product_review,Classify the sentiment of the text using exact...,The budget phone was excellent and worked smoo...,"{""label"": ""positive""}",pending,
3,slc_003,single_label_classification,product_review,Classify the sentiment of the text using exact...,The fitness watch was excellent and worked smo...,"{""label"": ""positive""}",pending,
4,slc_004,single_label_classification,product_review,Classify the sentiment of the text using exact...,The solar battery was excellent and worked smo...,"{""label"": ""positive""}",pending,


- multi_label_classification


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
50,mlc_050,multi_label_classification,institutional_announcement,Assign all applicable topic labels from {finan...,The government announced a new election reform...,"{""labels"": [""politics""]}",pending,
51,mlc_051,multi_label_classification,institutional_announcement,Assign all applicable topic labels from {finan...,The government announced a new cloud platform ...,"{""labels"": [""tech""]}",pending,
52,mlc_052,multi_label_classification,institutional_announcement,Assign all applicable topic labels from {finan...,The government announced a new mental health s...,"{""labels"": [""health""]}",pending,
53,mlc_053,multi_label_classification,institutional_announcement,Assign all applicable topic labels from {finan...,The government announced a new cycling related...,"{""labels"": [""sports""]}",pending,
54,mlc_054,multi_label_classification,institutional_announcement,Assign all applicable topic labels from {finan...,The government announced a new investment fund...,"{""labels"": [""finance""]}",pending,


- information_extraction


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
100,ie_100,information_extraction,meeting_announcement,"Extract the fields person, date, and location ...",Alice Smith will speak in Rome on 2024-01-15.,"{""person"": ""Alice Smith"", ""date"": ""2024-01-15""...",pending,
101,ie_101,information_extraction,meeting_announcement,"Extract the fields person, date, and location ...",Alice Smith will speak in Milan on 2024-03-21.,"{""person"": ""Alice Smith"", ""date"": ""2024-03-21""...",pending,
102,ie_102,information_extraction,meeting_announcement,"Extract the fields person, date, and location ...",Alice Smith will speak in Paris on 2024-06-10.,"{""person"": ""Alice Smith"", ""date"": ""2024-06-10""...",pending,
103,ie_103,information_extraction,meeting_announcement,"Extract the fields person, date, and location ...",Alice Smith will speak in Berlin on 2024-09-05.,"{""person"": ""Alice Smith"", ""date"": ""2024-09-05""...",pending,
104,ie_104,information_extraction,meeting_announcement,"Extract the fields person, date, and location ...",Alice Smith will speak in Madrid on 2025-02-14.,"{""person"": ""Alice Smith"", ""date"": ""2025-02-14""...",pending,


- rule_based_transformation


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
150,rbt_150,rule_based_transformation,mixed_case_statement__lowercase,Convert the text to lowercase.,Today Alice Smith Visited Rome With 3 Friends!,"{""text"": ""today alice smith visited rome with ...",pending,
151,rbt_151,rule_based_transformation,mixed_case_statement__remove_punctuation,Remove all punctuation from the text.,Today Bruno Rossi Visited Milan With 3 Friends!,"{""text"": ""Today Bruno Rossi Visited Milan With...",pending,
152,rbt_152,rule_based_transformation,mixed_case_statement__replace_numbers_with_NUM,Replace every number in the text with <NUM>.,Today Carla Gomez Visited Paris With 3 Friends!,"{""text"": ""Today Carla Gomez Visited Paris With...",pending,
153,rbt_153,rule_based_transformation,mixed_case_statement__remove_words_longer_than_6,Remove all words longer than 6 characters from...,Today David Lee Visited Berlin With 3 Friends!,"{""text"": ""Today David Lee Berlin With 3""}",pending,
154,rbt_154,rule_based_transformation,mixed_case_statement__lowercase,Convert the text to lowercase.,Today Elena Marino Visited Madrid With 3 Friends!,"{""text"": ""today elena marino visited madrid wi...",pending,


- extractive_qa


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
200,qa_200,extractive_qa,schedule_passage,Answer the question using an exact span from t...,PASSAGE: Alice Smith arrived in Rome on 2024-0...,"{""answer"": ""Rome""}",pending,
201,qa_201,extractive_qa,schedule_passage,Answer the question using an exact span from t...,PASSAGE: Alice Smith arrived in Milan on 2024-...,"{""answer"": ""Milan""}",pending,
202,qa_202,extractive_qa,schedule_passage,Answer the question using an exact span from t...,PASSAGE: Alice Smith arrived in Paris on 2024-...,"{""answer"": ""Paris""}",pending,
203,qa_203,extractive_qa,schedule_passage,Answer the question using an exact span from t...,PASSAGE: Alice Smith arrived in Berlin on 2024...,"{""answer"": ""Berlin""}",pending,
204,qa_204,extractive_qa,schedule_passage,Answer the question using an exact span from t...,PASSAGE: Alice Smith arrived in Madrid on 2025...,"{""answer"": ""Madrid""}",pending,


In [9]:
# Run automatic quality flags (These flags do not reject examples automatically, they only help identify suspicious cases for manual review)

flagged_rows = []

for ex in all_candidates:
    flags = auto_flag_candidate(ex)
    if flags:
        flagged_rows.append({
        "example_id": ex.example_id,
        "task_name": ex.task_name,
        "template_name": ex.template_name,
        "flags": ", ".join(flags),
        "rendered_input": render_input_for_review(ex),
        "gold_output": json.dumps(ex.gold_output, ensure_ascii=False)
        })

flagged_df = pd.DataFrame(flagged_rows)

print("Number of flagged examples:", len(flagged_df))
flagged_df.head(20)

Number of flagged examples: 0


""


In [10]:
# Check for exact duplicate inputs

duplicate_inputs = find_exact_duplicate_inputs(all_candidates)
print("Number of duplicate input strings:", len(duplicate_inputs))

Number of duplicate input strings: 0


In [11]:
duplicate_preview = [
{"rendered_input": text, "example_ids": ids}
for text, ids in list(duplicate_inputs.items())[:10]
]

duplicate_preview_df = pd.DataFrame(duplicate_preview)
duplicate_preview_df

""


In [12]:
# Approve a few examples manually as a demonstration

set_review_status(all_candidates, all_candidates[0].example_id, "approved", "Clear example.")
set_review_status(all_candidates, all_candidates[1].example_id, "approved", "Good wording.")
set_review_status(all_candidates, all_candidates[2].example_id, "rejected", "Too similar to another item.")

print("Approved:", len(get_examples_by_status(all_candidates, "approved")))
print("Rejected:", len(get_examples_by_status(all_candidates, "rejected")))
print("Pending:", len(get_examples_by_status(all_candidates, "pending")))

Approved: 2
Rejected: 1
Pending: 247


In [13]:
# Inspect reviewed examples

reviewed_rows = [candidate_to_review_row(ex) for ex in all_candidates]
reviewed_df = pd.DataFrame(reviewed_rows)

display(reviewed_df[reviewed_df["review_status"] != "pending"].head(20))

,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
0,slc_000,single_label_classification,product_review,Classify the sentiment of the text using exact...,The medical software was excellent and worked ...,"{""label"": ""positive""}",approved,Clear example.
1,slc_001,single_label_classification,product_review,Classify the sentiment of the text using exact...,The robot vacuum was excellent and worked smoo...,"{""label"": ""positive""}",approved,Good wording.
2,slc_002,single_label_classification,product_review,Classify the sentiment of the text using exact...,The budget phone was excellent and worked smoo...,"{""label"": ""positive""}",rejected,Too similar to another item.


In [14]:
# Save the full candidate pool 

save_candidates_to_jsonl(all_candidates, "data/candidates/candidates.jsonl")

# C:\code\testing\LLMs_Robustness_Under_Distractions\data\specs\benchmark_spec.json
# data\specs\benchmark_spec.json

In [15]:
# Debug: Remove this section later, just for inspecting duplicate inputs
duplicate_inputs = find_exact_duplicate_inputs(all_candidates)

for i, (text, ids) in enumerate(duplicate_inputs.items()):
    print("=" * 100)
    print(f"Duplicate group {i+1}")
    print("Example IDs:", ids)
    print("Rendered input:")
    print(text)
    print()
    
    if i == 9:   # show only first 10 groups for now
        break